# Build Repair Agent Evaluation

This notebook runs W&B Weave evaluations for the automatic build repair agent.

## Setup

In [ ]:
import os

import weave

os.environ["WEAVE_PARALLELISM"] = "4"

PROJECT_NAME = "bugbug-build-repair-eval"
_ = weave.init(PROJECT_NAME)

FIREFOX_REPO = os.environ["FIREFOX_GIT_REPO"]

## Load Dataset

In [ ]:
dataset = weave.ref("build_repair_one_commit_eval").get()
dataset = dataset.rows[:1]
print(f"Dataset has {len(dataset)} examples")

## Configure Model

In [ ]:
from scripts.build_repair_eval import BuildRepairModel

model = BuildRepairModel(
    firefox_repo=FIREFOX_REPO, analysis_only=True, no_try_push=True
)

## Run Evaluation

In [ ]:
from bugbug.tools.build_repair.scorer import (
    BasicMetricsScorer,
    BuildPassRateScorer,
)

evaluation = weave.Evaluation(
    name="build-repair-test",
    dataset=dataset,
    scorers=[
        BasicMetricsScorer(),
        BuildPassRateScorer(),
        # LLMFixMatchingScorer()
    ],
)

results = await evaluation.evaluate(model)

## Visualizations

In [ ]:
import matplotlib.pyplot as plt

basic = results.get("BasicMetricsScorer", {})
build = results.get("BuildPassRateScorer", {})

metrics = {
    "success_rate": basic.get("success_rate", 0),
    "diff_rate": basic.get("diff_rate", 0),
    "local_build_pass_rate": build.get("local_build_pass_rate", 0),
    "try_build_pass_rate": build.get("try_build_pass_rate", 0),
}

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(metrics.keys(), metrics.values())
ax.set_ylim(0, 1)
ax.set_ylabel("Rate")
ax.set_title("Build Repair Evaluation Results")
plt.tight_layout()
plt.show()

print(f"Total cost: ${basic.get('total_cost_usd', 0):.2f}")
print(f"Avg cost per example: ${basic.get('avg_cost_usd', 0):.2f}")

In [ ]:
print(f"Basic metrics: {basic}")
print(f"Build pass rates: {build}")

## 7. View in W&B

Visit [W&B Weave](https://wandb.ai) to see detailed traces, compare evaluations, and explore individual predictions.